# 04 · RAG Agent over Gutenberg (Sprint 5)

Локальный LLM (Ollama + qwen3:14b) с **tool calling** поверх 6 функций из `rag_tools.py`:

| Tool | Назначение |
|---|---|
| `semantic_search` | «найди упоминания X» → ChromaDB query (multilingual) |
| `corpus_stats_by_author` | «дай статистику по автору» |
| `top_ngrams_by_author` | «топ биграмм/триграмм» |
| `affinity_by_author` | «фирменные слова» |
| `word_contexts` | «в каком контексте слово X» |
| `compare_authors` | «сравни автора A и B» |

Модель сама выбирает инструмент. agentic loop max 5 итераций. См. [SESSION_2026-05-15-rag.md](../SESSION_2026-05-15-rag.md), [rag.md](../rag.md), [decisions.md](../decisions.md) D14/D15/D16.

In [5]:
import sys, json
sys.path.insert(0, '/workspace/scripts')
from rag_query import ask
from IPython.display import Markdown, display

def show(question, **kwargs):
    r = ask(question, **kwargs)
    md = f"### {question}\n\n{r['answer']}\n\n"
    md += f"<details><summary><b>Tool trace</b> · {r['iterations']} iter · {r['elapsed_sec']}s · {len(r['tool_calls'])} call(s)</summary>\n\n"
    for tc in r['tool_calls']:
        md += f"- **`{tc['name']}`** `{json.dumps(tc['args'], ensure_ascii=False)}`\n"
        md += f"  - → `{tc['result_summary']}`\n"
    md += "\n</details>"
    display(Markdown(md))
    return r

## 1 · Статистика автора → `corpus_stats_by_author`

In [6]:
_ = show('дай мне статистику по автору Достоевский')

[tool] corpus_stats_by_author done in 0.04s


### дай мне статистику по автору Достоевский

Вот статистика по автору Достоевский:

- **Количество книг**: 14
- **Количество книг с количеством токенов**: 13
- **Общее количество токенов**: 1,671,084
- **Уникальные слова**: 24,133
- **Средняя длина книги (в словах)**: 128,544
- **Самая длинная книга**: "The Brothers Karamazov" (PG28054), 352,925 токенов
- **Самая короткая книга**: "The Grand Inquisitor" (PG8578), 9,601 токенов
- **Языки**: английский

Достоевский представлен в корпусе Project Gutenberg в английском переводе.

<details><summary><b>Tool trace</b> · 2 iter · 4.41s · 1 call(s)</summary>

- **`corpus_stats_by_author`** `{"author_regex": "^Dostoyevsky,"}`
  - → `{"author_regex": "^Dostoyevsky,", "books_matched": 14, "books_with_counts": 13, "titles": ["The Brothers Karamazov", "The Possessed (The Devils)", "The Idiot", "Crime and Punishment", "The House of the Dead; or, Prison Life in Siberia: with...(truncated)`

</details>

## 2 · Биграммы → `top_ngrams_by_author(n=2)`

In [7]:
_ = show('какие самые популярные биграммы у Достоевского, дай таблицу топ-20')

[tool] top_ngrams_by_author(n=2) done in 1.98s, used 13 books, 1,219,559 ngrams


### какие самые популярные биграммы у Достоевского, дай таблицу топ-20

Вот таблица с топ-20 биграмм у Достоевского:

| Биграмма         | Количество упоминаний |
|------------------|----------------------|
| the prince       | 1656                 |
| to me            | 1524                 |
| at once          | 1481                 |
| as though        | 1465                 |
| of course        | 1463                 |
| there was        | 1455                 |
| all the          | 1441                 |
| you know         | 1363                 |
| the same         | 1348                 |
| to him           | 1287                 |
| the first        | 1035                 |
| him and          | 975                  |
| at last          | 955                  |
| to say           | 864                  |
| to see           | 845                  |
| the whole        | 839                  |
| he said          | 833                  |
| the old          | 821                  |
| at all           | 810                  |
| one of           | 809                  |

Эти биграммы отражают частые сочетания слов в текстах Достоевского, что может быть полезно для анализа его стиля и лексики.

<details><summary><b>Tool trace</b> · 2 iter · 7.24s · 1 call(s)</summary>

- **`top_ngrams_by_author`** `{"author_regex": "^Dostoyevsky,", "n": 2, "top": 20}`
  - → `{"author_regex": "^Dostoyevsky,", "n": 2, "books_used": 13, "total_ngrams": 1219559, "top": [{"ngram": "the prince", "count": 1656}, {"ngram": "to me", "count": 1524}, {"ngram": "at once", "count": 1481}, {"ngram": "as though", "count": 146...(truncated)`

</details>

## 3 · Триграммы → `top_ngrams_by_author(n=3)`

In [8]:
_ = show('топ-30 триграмм у Wodehouse')

[tool] top_ngrams_by_author(n=3) done in 3.31s, used 39 books, 1,840,457 ngrams


### топ-30 триграмм у Wodehouse

Вот топ-30 триграмм у P.G. Wodehouse, основанный на анализе его книг в корпусе Project Gutenberg:

| Триграмма             | Количество упоминаний |
|----------------------|----------------------|
| one of the           | 841                  |
| the fact that        | 565                  |
| sort of thing        | 494                  |
| do you know          | 451                  |
| one of those         | 446                  |
| the end of           | 433                  |
| it seemed to         | 397                  |
| do you mean          | 395                  |
| was one of           | 386                  |
| end of the           | 368                  |
| was going to         | 351                  |
| there was no         | 350                  |
| do you think         | 331                  |
| the sort of          | 325                  |
| that sort of         | 323                  |
| the first time       | 317                  |
| in new york          | 313                  |
| for the first        | 310                  |
| the rest of          | 310                  |
| and all that         | 309                  |
| in the world         | 309                  |
| mean to say          | 304                  |
| seemed to be         | 297                  |
| of the house         | 286                  |
| as far as            | 282                  |
| going to be          | 279                  |
| be able to           | 262                  |
| that there was       | 262                  |
| the young man        | 252                  |
| at the end           | 250                  |

Эти триграммы отражают характерные обороты речи и стилистику Wodehouse, часто используемые в его произведениях.

<details><summary><b>Tool trace</b> · 2 iter · 10.54s · 1 call(s)</summary>

- **`top_ngrams_by_author`** `{"author_regex": "^Wodehouse,", "n": 3, "top": 30}`
  - → `{"author_regex": "^Wodehouse,", "n": 3, "books_used": 39, "total_ngrams": 1840457, "top": [{"ngram": "one of the", "count": 841}, {"ngram": "the fact that", "count": 565}, {"ngram": "sort of thing", "count": 494}, {"ngram": "do you know", "...(truncated)`

</details>

## 4 · Фирменные слова → `affinity_by_author`

In [9]:
_ = show('фирменные слова Достоевского — топ-25')

[tool] running spgc_author_affinity.py for '^Dostoyevsky,' (slug=dostoyevsky)
[tool] affinity_by_author done in 1.86s (cached=False)


### фирменные слова Достоевского — топ-25

Вот топ-25 фирменных слов Достоевского, вычисленных по метрике affinity (частота у автора vs частота в корпусе):

| № | Слово             | Количество упоминаний у Достоевского | Количество в корпусе | Affinity |
|---|-------------------|--------------------------------------|----------------------|----------|
| 1 | polenka           | 44                                   | 44                   | 1702.09  |
| 2 | lebeziatnikov     | 83                                   | 83                   | 1702.09  |
| 3 | zossimov          | 86                                   | 86                   | 1702.09  |
| 4 | zametov           | 87                                   | 87                   | 1702.09  |
| 5 | svidrigaïlov      | 205                                  | 205                  | 1702.09  |
| 6 | resslich          | 11                                   | 11                   | 1702.09  |
| 7 | verhovensky       | 91                                   | 91                   | 1702.09  |
| 8 | virginsky         | 109                                  | 109                  | 1702.09  |
| 9 | surikoff          | 5                                    | 5                    | 1702.09  |
| 10| ptitsins          | 5                                    | 5                    | 1702.09  |
| 11| semeonovitch      | 5                                    | 5                    | 1702.09  |
| 12| gorohovaya        | 5                                    | 5                    | 1702.09  |
| 13| sokolovitch       | 5                                    | 5                    | 1702.09  |
| 14| roulettenberg     | 14                                   | 14                   | 1702.09  |
| 15| shlangenberg      | 11                                   | 11                   | 1702.09  |
| 16| burmergelm        | 5                                    | 5                    | 1702.09  |
| 17| bwikov            | 39                                   | 39                   | 1702.09  |
| 18| dievushkin        | 36                                   | 36                   | 1702.09  |
| 19| gorshkov          | 19                                   | 19                   | 1702.09  |
| 20| phaldoni          | 7                                    | 7                    | 1702.09  |
| 21| razumihin         | 344                                  | 344                  | 1702.09  |
| 22| kondratyevna      | 26                                   | 26                   | 1702.09  |
| 23| snegiryov         | 21                                   | 21                   | 1702.09  |
| 24| lyagavy           | 19                                   | 19                   | 1702.09  |
| 25| dardanelov        | 17                                   | 17                   | 1702.09  |

Эти слова часто встречаются в текстах Достоевского и могут быть использованы как маркеры его стиля. Например, имя "svidrigaïlov" — это персонаж из романа "Преступление и наказание", а "razumihin" — персонаж из "Преступления и наказания" и "Идиота".

<details><summary><b>Tool trace</b> · 2 iter · 16.16s · 1 call(s)</summary>

- **`affinity_by_author`** `{"author_regex": "^Dostoyevsky,", "min_author_count": 5, "top": 25}`
  - → `{"author_regex": "^Dostoyevsky,", "slug": "dostoyevsky", "total_unique_words": 10809, "top": [{"word": "polenka", "author_count": 44, "corpus_count": 44, "affinity": 1702.09}, {"word": "lebeziatnikov", "author_count": 83, "corpus_count": 83...(truncated)`

</details>

## 5 · Контексты слова → `word_contexts`

In [ ]:
_ = show("в каких контекстах Достоевский использует слово 'misery'? приведи 5 примеров")

## 6 · Сравнение авторов → `compare_authors`

In [ ]:
_ = show('сравни Достоевского и Толстого по фирменным словам')

## 7 · Семантический поиск без фильтра → `semantic_search`

In [ ]:
_ = show('найди все упоминания битой посуды в книгах')

## 8 · Семантический поиск с author_filter → `semantic_search(author_filter=...)`

In [ ]:
_ = show('найди упоминания самоубийства у Достоевского')

## 9 · Английский запрос → top_ngrams

In [ ]:
_ = show("what are Wodehouse's signature exclamations? show top-15 bigrams")

## 10 · Word contexts у другого автора

In [ ]:
_ = show("у Сервантеса есть слово 'donkey'? приведи контексты")

## ⚙ Произвольный запрос

In [ ]:
_ = show('сравни Wodehouse и Doyle')